# Notebook 1 - API et nettoyage

Dans ce notebook, nous récupèrons les données, harmonisons les régions et préparons le dataset final.

## APIs utilisées

- `SNCF Open Data - regularite-mensuelle-ter` : base principale du projet, avec la régularité mensuelle TER par région depuis 2013.
- `SNCF Open Data - ponctualite-mensuelle-transilien` : comparaison avec le réseau Transilien en Île-de-France.
- `SNCF Open Data - regularite-mensuelle-intercites` : comparaison avec les liaisons Intercités longue distance.
- `SNCF Open Data - gares-de-voyageurs` : référentiel des gares pour construire une variable de contexte réseau.
- `SNCF Open Data - frequentation-gares` : fréquentation annuelle des gares pour mesurer la charge du réseau.
- `geo.api.gouv.fr` : régions et départements officiels pour éviter un mapping statique trop lourd.
- `Open-Meteo Archive API` : historique météo quotidien pour construire notre score de stress météo.

Ici, on prépare trois briques : la base `TER + météo`, une base `comparaison des modes`, et une base `contexte réseau`.


In [1]:
from pathlib import Path
import math
import re
import textwrap
import time
import unicodedata
from typing import Iterable, List

import numpy as np
import pandas as pd
import requests

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = PROJECT_DIR / "figures"

TER_RAW_PATH = RAW_DIR / "sncf_ter_regularite_mensuelle.csv"
TRANSILIEN_RAW_PATH = RAW_DIR / "sncf_transilien_regularite_mensuelle.csv"
INTERCITES_RAW_PATH = RAW_DIR / "sncf_intercites_regularite_mensuelle.csv"
GARES_RAW_PATH = RAW_DIR / "sncf_gares_voyageurs.csv"
FREQUENTATION_RAW_PATH = RAW_DIR / "sncf_frequentation_gares.csv"
GEO_REGIONS_CACHE_PATH = RAW_DIR / "geo_api_current_regions_reference.csv"
GEO_DEPARTEMENTS_CACHE_PATH = RAW_DIR / "geo_api_departements_reference.csv"
WEATHER_CACHE_PATH = RAW_DIR / "open_meteo_daily_current_regions.csv"
TER_MONTHLY_PATH = PROCESSED_DIR / "ter_current_regions_monthly.csv"
WEATHER_MONTHLY_PATH = PROCESSED_DIR / "weather_current_regions_monthly.csv"
TRANSILIEN_MONTHLY_PATH = PROCESSED_DIR / "transilien_monthly.csv"
INTERCITES_MONTHLY_PATH = PROCESSED_DIR / "intercites_monthly.csv"
MODE_COMPARISON_PATH = PROCESSED_DIR / "rail_mode_comparison_monthly.csv"
REGION_CONTEXT_PATH = PROCESSED_DIR / "rail_region_context.csv"
MERGED_PATH = PROCESSED_DIR / "ter_weather_current_regions_monthly.csv"

ANALYSIS_START_DATE = "2013-01-01"
ANALYSIS_END_DATE = "2026-02-28"

NUMERIC_TER_COLUMNS = [
    "nombre_de_trains_programmes",
    "nombre_de_trains_ayant_circule",
    "nombre_de_trains_annules",
    "nombre_de_trains_en_retard_a_l_arrivee",
]

NUMERIC_INTERCITES_COLUMNS = [
    "nombre_de_trains_programmes",
    "nombre_de_trains_ayant_circule",
    "nombre_de_trains_annules",
    "nombre_de_trains_en_retard_a_l_arrivee",
    "taux_de_regularite",
]

METRO_REGION_NAMES = {
    "Auvergne-Rhône-Alpes",
    "Bourgogne-Franche-Comté",
    "Bretagne",
    "Centre-Val de Loire",
    "Corse",
    "Grand Est",
    "Hauts-de-France",
    "Île-de-France",
    "Normandie",
    "Nouvelle-Aquitaine",
    "Occitanie",
    "Pays de la Loire",
    "Provence-Alpes-Côte d'Azur",
}

STRESS_COMPONENTS = {
    "heavy_rain_days": "Pluie forte",
    "very_heavy_rain_days": "Pluie très forte",
    "snow_days": "Neige",
    "strong_wind_days": "Vent fort",
    "heat_days": "Chaleur",
    "frost_days": "Gel",
    "storm_days": "Orage",
}

SNCF_DATASET_URLS = {
    TER_RAW_PATH: "https://ressources.data.sncf.com/explore/dataset/regularite-mensuelle-ter/download/?format=csv&timezone=Europe/Paris&lang=fr&use_labels_for_header=true&csv_separator=%3B",
    TRANSILIEN_RAW_PATH: "https://ressources.data.sncf.com/explore/dataset/ponctualite-mensuelle-transilien/download/?format=csv&timezone=Europe/Paris&lang=fr&use_labels_for_header=true&csv_separator=%3B",
    INTERCITES_RAW_PATH: "https://ressources.data.sncf.com/explore/dataset/regularite-mensuelle-intercites/download/?format=csv&timezone=Europe/Paris&lang=fr&use_labels_for_header=true&csv_separator=%3B",
    GARES_RAW_PATH: "https://ressources.data.sncf.com/explore/dataset/gares-de-voyageurs/download/?format=csv&timezone=Europe/Paris&lang=fr&use_labels_for_header=true&csv_separator=%3B",
    FREQUENTATION_RAW_PATH: "https://ressources.data.sncf.com/explore/dataset/frequentation-gares/download/?format=csv&timezone=Europe/Paris&lang=fr&use_labels_for_header=true&csv_separator=%3B",
}


def ensure_directories():
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    RAW_DIR.mkdir(parents=True, exist_ok=True)


def normalize_text(value):
    text = unicodedata.normalize("NFKD", str(value or ""))
    ascii_text = text.encode("ascii", "ignore").decode("ascii")
    ascii_text = ascii_text.replace("-", " ").replace("'", " ").replace("/", " ")
    return " ".join(ascii_text.split()).lower()


def normalize_columns(columns):
    return [normalize_text(column).replace(" ", "_") for column in columns]


def to_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def weighted_average(values, weights):
    mask = values.notna() & weights.notna()
    if not mask.any():
        return math.nan
    return float(np.average(values[mask], weights=weights[mask]))


def fetch_current_regions_reference(force_refresh=False):
    if GEO_REGIONS_CACHE_PATH.exists() and not force_refresh:
        df = pd.read_csv(GEO_REGIONS_CACHE_PATH)
        return df[df["region_current"].isin(METRO_REGION_NAMES)].copy()

    regions = requests.get("https://geo.api.gouv.fr/regions", timeout=30).json()
    rows = []
    for region in regions:
        detail = requests.get(
            f"https://geo.api.gouv.fr/regions/{region['code']}?fields=nom,code,chefLieu",
            timeout=30,
        ).json()
        chef_lieu_code = detail.get("chefLieu")
        if not chef_lieu_code:
            continue
        commune = requests.get(
            f"https://geo.api.gouv.fr/communes/{chef_lieu_code}?fields=nom,centre",
            timeout=30,
        ).json()
        coords = (commune.get("centre") or {}).get("coordinates") or [None, None]
        lon, lat = coords[0], coords[1]
        rows.append(
            {
                "region_code": detail["code"],
                "region_current": detail["nom"],
                "representative_city": commune.get("nom"),
                "latitude": lat,
                "longitude": lon,
            }
        )
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    df = df[df["region_current"].isin(METRO_REGION_NAMES)].copy().sort_values("region_current")
    df.to_csv(GEO_REGIONS_CACHE_PATH, index=False)
    return df


def fetch_departements_reference(force_refresh=False):
    if GEO_DEPARTEMENTS_CACHE_PATH.exists() and not force_refresh:
        return pd.read_csv(GEO_DEPARTEMENTS_CACHE_PATH)

    departments = requests.get(
        "https://geo.api.gouv.fr/departements?fields=nom,code,region",
        timeout=30,
    ).json()
    rows = []
    for department in departments:
        region = department.get("region") or {}
        rows.append(
            {
                "departement_code": department.get("code"),
                "departement_name": department.get("nom"),
                "region_code": region.get("code"),
                "region_current": region.get("nom"),
            }
        )
    df = pd.DataFrame(rows)
    if "20" not in set(df["departement_code"]):
        df = pd.concat(
            [df, pd.DataFrame([{"departement_code": "20", "departement_name": "Corse", "region_code": "94", "region_current": "Corse"}])],
            ignore_index=True,
        )
    df.to_csv(GEO_DEPARTEMENTS_CACHE_PATH, index=False)
    return df


In [2]:
SNCF_REGION_ALIASES = {
    "Alsace": "Grand Est",
    "Lorraine": "Grand Est",
    "Champagne Ardenne": "Grand Est",
    "Aquitaine": "Nouvelle-Aquitaine",
    "Limousin": "Nouvelle-Aquitaine",
    "Poitou Charentes": "Nouvelle-Aquitaine",
    "Auvergne": "Auvergne-Rhône-Alpes",
    "Rhône Alpes": "Auvergne-Rhône-Alpes",
    "Rhone Alpes": "Auvergne-Rhône-Alpes",
    "Bourgogne": "Bourgogne-Franche-Comté",
    "Franche Comté": "Bourgogne-Franche-Comté",
    "Languedoc Roussillon": "Occitanie",
    "Midi Pyrénées": "Occitanie",
    "Midi Pyrenees": "Occitanie",
    "Basse Normandie": "Normandie",
    "Haute Normandie": "Normandie",
    "Nord Pas de Calais": "Hauts-de-France",
    "Picardie": "Hauts-de-France",
    "Etoile Amiens": "Hauts-de-France",
    "Centre": "Centre-Val de Loire",
    "Centre Val-de-Loire": "Centre-Val de Loire",
    "Pays-de-la-Loire": "Pays de la Loire",
    "Loire Océan": "Pays de la Loire",
    "Loire Ocean": "Pays de la Loire",
    "Nouvelle Aquitaine": "Nouvelle-Aquitaine",
    "Sud Azur": "Provence-Alpes-Côte d'Azur",
    "Provence Alpes Cote d'Azur": "Provence-Alpes-Côte d'Azur",
    "Ile de France": "Île-de-France",
    "Ile-de-France": "Île-de-France",
    "PACA": "Provence-Alpes-Côte d'Azur",
}


def build_region_lookup(region_reference):
    lookup = {
        normalize_text(region_name): region_name
        for region_name in region_reference["region_current"].dropna().unique()
    }
    lookup.update({normalize_text(old): new for old, new in SNCF_REGION_ALIASES.items()})
    return lookup


def zscore_by_region_month(series):
    std = float(series.std(ddof=0))
    if math.isclose(std, 0.0) or np.isnan(std):
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - float(series.mean())) / std


def shorten_comment(text, width=140):
    if pd.isna(text) or not str(text).strip():
        return "Pas de commentaire SNCF disponible."
    clean = " ".join(str(text).split())
    return textwrap.shorten(clean, width=width, placeholder="...")


def clean_uic(value):
    text = str(value or "").strip()
    if not text or text.lower() == "nan":
        return None
    digits = re.sub(r"\D", "", text)
    return digits or None


def split_uic_values(value):
    text = str(value or "").strip()
    if not text or text.lower() == "nan":
        return []
    parts = re.split(r"[,;/]", text)
    return [code for code in (clean_uic(part) for part in parts) if code]


def extract_department_code(code_commune):
    text = str(code_commune or "").strip().upper().replace(" ", "")
    if not text or text == "NAN":
        return None
    if text.startswith(("97", "98")):
        return text[:3]
    if text.startswith(("2A", "2B")):
        return text[:2]
    if text.startswith("20"):
        return "20"
    return text[:2]


def download_if_needed(path, url, force_refresh=False):
    if path.exists() and not force_refresh:
        return
    response = requests.get(url, timeout=180)
    response.raise_for_status()
    path.write_bytes(response.content)


## Réglages

In [3]:
ensure_directories()

DOWNLOAD_SNCF_FROM_API = False
FORCE_GEO_REFRESH = False
FORCE_WEATHER_REFRESH = False

TER_CSV_URL = (
    "https://ressources.data.sncf.com/explore/dataset/regularite-mensuelle-ter/"
    "download/?format=csv&timezone=Europe/Berlin&lang=fr&use_labels_for_header=true"
    "&csv_separator=%3B"
)

if DOWNLOAD_SNCF_FROM_API:
    response = requests.get(TER_CSV_URL, timeout=60)
    response.raise_for_status()
    TER_RAW_PATH.write_bytes(response.content)

region_reference = fetch_current_regions_reference(force_refresh=FORCE_GEO_REFRESH)
region_lookup = build_region_lookup(region_reference)

print("CSV SNCF :", TER_RAW_PATH)
print("Cache régions API :", GEO_REGIONS_CACHE_PATH)
print("Cache météo :", WEATHER_CACHE_PATH)
region_reference.head(12)

CSV SNCF : c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\raw\sncf_ter_regularite_mensuelle.csv
Cache régions API : c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\raw\geo_api_current_regions_reference.csv
Cache météo : c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\raw\open_meteo_daily_current_regions.csv


,region_code,region_current,representative_city,latitude,longitude
2,53,Bretagne,Rennes,48.1159,-1.6884
3,24,Centre-Val de Loire,Orléans,47.8734,1.9122
4,94,Corse,Ajaccio,41.9228,8.7058
5,44,Grand Est,Strasbourg,48.5691,7.7621
6,32,Hauts-de-France,Lille,50.6311,3.0468
8,28,Normandie,Rouen,49.4412,1.0912
9,75,Nouvelle-Aquitaine,Bordeaux,44.8624,-0.5848
10,76,Occitanie,Toulouse,43.6007,1.4328
11,52,Pays de la Loire,Nantes,47.2382,-1.5603


## TER

In [4]:
def load_ter_monthly(region_lookup):
    raw = pd.read_csv(TER_RAW_PATH, sep=";", encoding="utf-8-sig")
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")
    raw["region_current"] = raw["region"].map(lambda value: region_lookup.get(normalize_text(value)))
    raw = raw.dropna(subset=["date", "region_current"]).copy()

    rows = []
    for (date, region_current), group in raw.groupby(["date", "region_current"], sort=True):
        row = {
            "date": date,
            "region_current": region_current,
            "source_regions": ", ".join(sorted(group["region"].dropna().unique())),
        }
        for column in NUMERIC_TER_COLUMNS:
            row[column] = np.nan if group[column].isna().any() else float(group[column].sum())
        comments = [str(value) for value in group["commentaires"].dropna().unique() if str(value).strip()]
        row["commentaires"] = " | ".join(comments)
        rows.append(row)

    ter = pd.DataFrame(rows).sort_values(["date", "region_current"]).reset_index(drop=True)
    ter["regularity_pct"] = 100 * (ter["nombre_de_trains_ayant_circule"] - ter["nombre_de_trains_en_retard_a_l_arrivee"]) / ter["nombre_de_trains_ayant_circule"]
    ter["cancellation_pct"] = 100 * ter["nombre_de_trains_annules"] / ter["nombre_de_trains_programmes"]
    ter["delay_pct"] = 100 * ter["nombre_de_trains_en_retard_a_l_arrivee"] / ter["nombre_de_trains_ayant_circule"]
    ter["month"] = ter["date"].dt.month
    ter["year"] = ter["date"].dt.year
    ter["regularity_gap"] = ter["regularity_pct"] - ter.groupby(["region_current", "month"])["regularity_pct"].transform("mean")
    ter["cancellation_gap"] = ter["cancellation_pct"] - ter.groupby(["region_current", "month"])["cancellation_pct"].transform("mean")
    ter.to_csv(TER_MONTHLY_PATH, index=False)
    return ter


ter_monthly = load_ter_monthly(region_lookup)
ter_monthly.head(10)

,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,delay_pct,month,year,regularity_gap,cancellation_gap
0,2013-01-01,Auvergne-Rhône-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,10.909041,1,2013,0.259094,0.334447
1,2013-01-01,Bourgogne-Franche-Comté,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,8.368855,1,2013,-0.346137,-0.013286
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,6.418723,1,2013,-1.342716,0.126337
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,8.382368,1,2013,0.544582,0.042994
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,NaN,1,2013,NaN,NaN
5,2013-01-01,Hauts-de-France,"Nord Pas de Calais, Picardie",30981.0,30563.0,418.0,3577.0,Fortes chutes de neige ayant entrainé des pert...,88.296306,1.349214,11.703694,1,2013,-1.826960,-1.258907
6,2013-01-01,Normandie,"Basse Normandie, Haute Normandie",9288.0,9175.0,113.0,799.0,Grand froid et épisode neigeux les semaines 3 ...,91.291553,1.216624,8.708447,1,2013,-0.988505,-0.808950
7,2013-01-01,Nouvelle-Aquitaine,"Aquitaine, Limousin, Poitou Charentes",15185.0,14918.0,267.0,1146.0,Intempéries à partir du 20 janvier. | Mouvemen...,92.318005,1.758314,7.681995,1,2013,1.728197,-0.938219
8,2013-01-01,Occitanie,"Languedoc Roussillon, Midi Pyrénées",13232.0,12838.0,394.0,1280.0,Chute de neige sur le littoral.,90.029600,2.977630,9.970400,1,2013,1.205988,-0.240586
9,2013-01-01,Pays de la Loire,Pays-de-la-Loire,10407.0,10195.0,212.0,713.0,,93.006376,2.037090,6.993624,1,2013,0.864085,-0.236635


In [5]:
target_regions = sorted(ter_monthly["region_current"].dropna().unique())
region_reference_target = region_reference[region_reference["region_current"].isin(target_regions)].copy()

print("Régions gardées :")
print(target_regions)
region_reference_target

Régions gardées :
['Auvergne-Rhône-Alpes', 'Bourgogne-Franche-Comté', 'Bretagne', 'Centre-Val de Loire', 'Grand Est', 'Hauts-de-France', 'Normandie', 'Nouvelle-Aquitaine', 'Occitanie', 'Pays de la Loire', "Provence-Alpes-Côte d'Azur"]


,region_code,region_current,representative_city,latitude,longitude
2,53,Bretagne,Rennes,48.1159,-1.6884
3,24,Centre-Val de Loire,Orléans,47.8734,1.9122
5,44,Grand Est,Strasbourg,48.5691,7.7621
6,32,Hauts-de-France,Lille,50.6311,3.0468
8,28,Normandie,Rouen,49.4412,1.0912
9,75,Nouvelle-Aquitaine,Bordeaux,44.8624,-0.5848
10,76,Occitanie,Toulouse,43.6007,1.4328
11,52,Pays de la Loire,Nantes,47.2382,-1.5603


## Météo

In [6]:
def request_weather_archive(url, attempts=6):
    for attempt in range(attempts):
        response = requests.get(url, timeout=180)
        if response.status_code == 200:
            return response.json()
        if response.status_code in {429, 500, 502, 503, 504}:
            time.sleep(15 * (attempt + 1))
            continue
        response.raise_for_status()
    raise RuntimeError("La requête météo a échoué après plusieurs tentatives.")


def load_weather_daily(region_reference_target, region_lookup, force_refresh=False):
    if WEATHER_CACHE_PATH.exists() and not force_refresh:
        daily = pd.read_csv(WEATHER_CACHE_PATH)
        daily["date"] = pd.to_datetime(daily["date"])
        daily["region_current"] = daily["region_current"].map(lambda value: region_lookup.get(normalize_text(value), value))
        daily = daily[daily["region_current"].isin(region_reference_target["region_current"])].copy()
        if daily["region_current"].nunique() == region_reference_target["region_current"].nunique():
            return daily

    latitudes = ",".join(region_reference_target["latitude"].astype(str))
    longitudes = ",".join(region_reference_target["longitude"].astype(str))
    url = (
        "https://archive-api.open-meteo.com/v1/archive?"
        f"latitude={latitudes}&longitude={longitudes}"
        "&start_date=2013-01-01&end_date=2026-02-28"
        "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
        "precipitation_sum,snowfall_sum,wind_speed_10m_max,wind_gusts_10m_max,weather_code"
        "&timezone=Europe%2FParis"
    )
    payloads = request_weather_archive(url)

    frames = []
    rows = region_reference_target.reset_index(drop=True)
    for idx, payload in enumerate(payloads):
        meta = rows.loc[idx]
        frame = pd.DataFrame(payload["daily"])
        frame["date"] = pd.to_datetime(frame["time"])
        frame["region_current"] = meta["region_current"]
        frame["representative_city"] = meta["representative_city"]
        frames.append(frame.drop(columns=["time"]))

    daily = pd.concat(frames, ignore_index=True)
    daily.to_csv(WEATHER_CACHE_PATH, index=False)
    return daily


weather_daily = load_weather_daily(region_reference_target, region_lookup, force_refresh=FORCE_WEATHER_REFRESH)
weather_daily.head()

,temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,snowfall_sum,wind_speed_10m_max,wind_gusts_10m_max,weather_code,date,region_current,representative_city
0,7.4,9.0,4.0,8.5,0.0,24.4,49.3,55,2013-01-01,Centre-Val de Loire,Orléans
1,4.6,7.9,1.6,0.0,0.0,10.0,18.7,3,2013-01-02,Centre-Val de Loire,Orléans
2,8.2,9.6,6.6,0.2,0.0,13.0,22.0,51,2013-01-03,Centre-Val de Loire,Orléans
3,8.6,10.0,6.8,0.0,0.0,8.0,16.6,3,2013-01-04,Centre-Val de Loire,Orléans
4,8.2,9.1,7.5,0.0,0.0,9.1,18.4,3,2013-01-05,Centre-Val de Loire,Orléans


In [7]:
def build_weather_monthly(daily):
    weather = daily.copy()
    weather["month_start"] = weather["date"].dt.to_period("M").dt.to_timestamp()
    weather["month"] = weather["date"].dt.month

    weather["heavy_rain_day"] = (weather["precipitation_sum"] >= 10).astype(int)
    weather["very_heavy_rain_day"] = (weather["precipitation_sum"] >= 20).astype(int)
    weather["snow_day"] = (weather["snowfall_sum"] > 0.1).astype(int)
    weather["strong_wind_day"] = (weather["wind_gusts_10m_max"] >= 70).astype(int)
    weather["heat_day"] = (weather["temperature_2m_max"] >= 30).astype(int)
    weather["frost_day"] = (weather["temperature_2m_min"] <= 0).astype(int)
    weather["storm_day"] = weather["weather_code"].isin([95, 96, 99]).astype(int)

    monthly = (
        weather.groupby(["month_start", "region_current"], as_index=False)
        .agg(
            representative_city=("representative_city", "first"),
            temp_mean=("temperature_2m_mean", "mean"),
            temp_max_peak=("temperature_2m_max", "max"),
            temp_min_low=("temperature_2m_min", "min"),
            precipitation_total=("precipitation_sum", "sum"),
            snowfall_total=("snowfall_sum", "sum"),
            heavy_rain_days=("heavy_rain_day", "sum"),
            very_heavy_rain_days=("very_heavy_rain_day", "sum"),
            snow_days=("snow_day", "sum"),
            strong_wind_days=("strong_wind_day", "sum"),
            heat_days=("heat_day", "sum"),
            frost_days=("frost_day", "sum"),
            storm_days=("storm_day", "sum"),
        )
        .rename(columns={"month_start": "date"})
    )

    monthly["month"] = monthly["date"].dt.month
    monthly["year"] = monthly["date"].dt.year

    for metric in STRESS_COMPONENTS:
        monthly[f"{metric}_z"] = monthly.groupby(["region_current", "month"])[metric].transform(zscore_by_region_month)
        monthly[f"{metric}_stress"] = monthly[f"{metric}_z"].clip(lower=0)

    stress_columns = [f"{metric}_stress" for metric in STRESS_COMPONENTS]
    monthly["weather_stress_score"] = monthly[stress_columns].mean(axis=1)
    monthly["dominant_weather_key"] = monthly[stress_columns].idxmax(axis=1).str.replace("_stress", "", regex=False)
    monthly["dominant_weather_label"] = monthly["dominant_weather_key"].map(STRESS_COMPONENTS)
    monthly.to_csv(WEATHER_MONTHLY_PATH, index=False)
    return monthly


weather_monthly = build_weather_monthly(weather_daily)
weather_monthly.head()

,date,region_current,representative_city,temp_mean,temp_max_peak,temp_min_low,precipitation_total,snowfall_total,heavy_rain_days,very_heavy_rain_days,...,strong_wind_days_stress,heat_days_z,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label
0,2013-01-01,Bretagne,Rennes,5.248387,13.6,-3.6,65.4,6.51,0,0,...,0.0,0.0,0.0,0.404067,0.404067,0.0,0.0,0.339054,snow_days,Neige
1,2013-01-01,Centre-Val de Loire,Orléans,3.780645,13.2,-5.2,63.3,13.51,2,0,...,0.0,0.0,0.0,0.550090,0.550090,0.0,0.0,0.412076,snow_days,Neige
2,2013-01-01,Grand Est,Strasbourg,1.722581,12.9,-6.5,62.6,8.40,2,0,...,0.0,0.0,0.0,0.438215,0.438215,0.0,0.0,0.155910,snow_days,Neige
3,2013-01-01,Hauts-de-France,Lille,2.406452,13.0,-9.9,53.8,13.72,0,0,...,0.0,0.0,0.0,0.904748,0.904748,0.0,0.0,0.421994,snow_days,Neige
4,2013-01-01,Normandie,Rouen,4.080645,13.7,-4.3,55.8,13.37,0,0,...,0.0,0.0,0.0,0.504695,0.504695,0.0,0.0,0.406794,snow_days,Neige


## Fusion

In [8]:
def merge_datasets(ter, weather_monthly):
    merged = ter.merge(weather_monthly, on=["date", "region_current", "month", "year"], how="left")
    q20 = merged["weather_stress_score"].quantile(0.20)
    q80 = merged["weather_stress_score"].quantile(0.80)
    merged["stress_bucket"] = pd.cut(
        merged["weather_stress_score"],
        bins=[-1e9, q20, q80, 1e9],
        labels=["Mois calmes", "Mois intermédiaires", "Mois chocs météo"],
        include_lowest=True,
    )
    merged["comment_clean"] = merged["commentaires"].map(shorten_comment)
    merged.to_csv(MERGED_PATH, index=False)
    return merged


merged = merge_datasets(ter_monthly, weather_monthly)
print("Exports créés :")
for path_out in [TER_MONTHLY_PATH, WEATHER_MONTHLY_PATH, MERGED_PATH]:
    print('-', path_out)
merged.head()

Exports créés :
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\ter_current_regions_monthly.csv
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\weather_current_regions_monthly.csv
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\ter_weather_current_regions_monthly.csv


,date,region_current,source_regions,nombre_de_trains_programmes,nombre_de_trains_ayant_circule,nombre_de_trains_annules,nombre_de_trains_en_retard_a_l_arrivee,commentaires,regularity_pct,cancellation_pct,...,heat_days_stress,frost_days_z,frost_days_stress,storm_days_z,storm_days_stress,weather_stress_score,dominant_weather_key,dominant_weather_label,stress_bucket,comment_clean
0,2013-01-01,Auvergne-Rhône-Alpes,"Auvergne, Rhône Alpes",37223.0,36511.0,712.0,3983.0,Conditions météos défavorables.,89.090959,1.912796,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Conditions météos défavorables.
1,2013-01-01,Bourgogne-Franche-Comté,"Bourgogne, Franche Comté",14226.0,14076.0,150.0,1178.0,Un mois de janvier qui surpasse les six exerci...,91.631145,1.054407,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Un mois de janvier qui surpasse les six exerci...
2,2013-01-01,Bretagne,Bretagne,8776.0,8631.0,145.0,554.0,Fortes chutes de neige ayant entrainé des pert...,93.581277,1.652233,...,0.0,0.404067,0.404067,0.0,0.0,0.339054,snow_days,Neige,Mois intermédiaires,Fortes chutes de neige ayant entrainé des pert...
3,2013-01-01,Centre-Val de Loire,Centre,9882.0,9687.0,195.0,812.0,"Trois incidents caténaires lourds, dont deux p...",91.617632,1.973285,...,0.0,0.550090,0.550090,0.0,0.0,0.412076,snow_days,Neige,Mois intermédiaires,"Trois incidents caténaires lourds, dont deux p..."
4,2013-01-01,Grand Est,"Alsace, Champagne Ardenne, Lorraine",NaN,NaN,NaN,NaN,Conditions météorologiques dégradées du 15 au ...,NaN,NaN,...,0.0,0.438215,0.438215,0.0,0.0,0.155910,snow_days,Neige,Mois intermédiaires,Conditions météorologiques dégradées du 15 au ...


In [9]:
print("Exports créés :")
for path in [TER_MONTHLY_PATH, WEATHER_MONTHLY_PATH, MERGED_PATH]:
    print('-', path)

Exports créés :
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\ter_current_regions_monthly.csv
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\weather_current_regions_monthly.csv
- c:\Users\antoc\analyse-impact-meteo-regularite-sncf\data\processed\ter_weather_current_regions_monthly.csv


In [10]:
merged[[
    'date',
    'region_current',
    'regularity_pct',
    'cancellation_pct',
    'weather_stress_score',
    'dominant_weather_label',
    'stress_bucket'
]].sample(12, random_state=42).sort_values(['date', 'region_current'])

,date,region_current,regularity_pct,cancellation_pct,weather_stress_score,dominant_weather_label,stress_bucket
561,2017-04-01,Auvergne-Rhône-Alpes,90.049054,1.235911,NaN,NaN,NaN
567,2017-04-01,Normandie,95.407750,0.643604,0.000000,Pluie forte,Mois calmes
582,2017-05-01,Provence-Alpes-Côte d'Azur,84.508052,2.206211,NaN,NaN,NaN
1098,2022-01-01,Occitanie,89.878954,4.174256,0.367908,Gel,Mois intermédiaires
1156,2022-08-01,Centre-Val de Loire,92.261514,1.085545,0.208941,Chaleur,Mois intermédiaires
1202,2023-01-01,Grand Est,92.976505,2.960041,0.067763,Vent fort,Mois intermédiaires
1253,2023-06-01,Provence-Alpes-Côte d'Azur,85.849910,2.690813,NaN,NaN,NaN
1298,2023-11-01,Provence-Alpes-Côte d'Azur,87.316633,3.182811,NaN,NaN,NaN
1320,2024-02-01,Hauts-de-France,89.245330,2.861373,0.211014,Pluie forte,Mois intermédiaires
1322,2024-02-01,Nouvelle-Aquitaine,91.693110,2.642467,0.407312,Pluie très forte,Mois intermédiaires


Ce fichier fusionné sert ensuite à notre notebook d'analyse.